# Appendix — Self-tests and Sanity Checks

*Written for idtrack 0.1.2.*

Use this appendix to **confirm that your setup is correct** after building graphs (Part 2). For each of the five organisms it checks:
- which externals configuration a build uses (the local repository's or the shipped one);
- that the graph loads, and what it holds;
- that a gene id of the snapshot release converts to itself;

and, optionally, runs one of the deeper self-tests of `idtrack.checks.GraphChecks`.

> **Tip:** If something fails, see section 7.3 of Part 7 for a diagnostic checklist.

## A.1 — Setup

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import API, ids

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)
DEFAULT_CONFIG = Path(idtrack.__file__).parent / 'default_config'
SNAPSHOT_RELEASE = 115

API.configure_logger()
print('Local repository:', LOCAL_REPOSITORY)

Local repository: idtrack_cache


## A.2 — Helper: a gene id of the snapshot release

So that no gene id is typed by hand, the smoke test takes, from each graph, the first base Ensembl gene id (in the order of the graph's nodes) whose latest version is active at the snapshot release (ids look like `ENSG...`, `ENSMUSG...`, `ENSSSCG...`, `ENSMMUG...`, `ENSMFAG...`). Converted from and to the snapshot release, such an id must answer its own current version.

In [2]:
def gene_id_of_snapshot(graph, snapshot_release) -> str:
    """The first base Ensembl gene id in node order whose latest version is active at the snapshot release."""
    for node in graph.nodes:
        if graph.node_type(node) != ids.BASE_GENE:
            continue
        latest = graph.versions_of(node)[-1]
        if any(start <= snapshot_release <= end for start, end in graph.active_ranges(latest)):
            return str(node)
    raise RuntimeError('No gene id active at the snapshot release.')

## A.3 — The checklist, per organism

For each organism: the configuration a build uses, the graph, and the smoke conversion.

In [3]:
for organism in ['homo_sapiens', 'mus_musculus', 'sus_scrofa', 'macaca_mulatta', 'macaca_fascicularis']:
    name = f'{organism}_externals_modified.yml'
    configuration = 'local repository' if (LOCAL_REPOSITORY / name).exists() else 'shipped'
    api = API.build(organism, SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))  # loads the graph file of Part 2
    described = api.describe()
    example_id = gene_id_of_snapshot(api.graph, SNAPSHOT_RELEASE)
    result = api.convert(example_id, from_release=SNAPSHOT_RELEASE, to_release=SNAPSHOT_RELEASE)
    print(f'{organism:<20} configuration: {configuration:<16} primary assembly {described.primary_assembly:<4} '
          f'releases {described.releases[0]}-{described.releases[-1]} | {example_id} -> {result.status} {list(result.targets())}')

2026-11-24 11:54:14 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


homo_sapiens         configuration: local repository primary assembly 38   releases 48-115 | ENSG00000000003 -> converted ['ENSG00000000003.17']


2026-11-24 11:54:18 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


mus_musculus         configuration: local repository primary assembly 39   releases 48-115 | ENSMUSG00000000001 -> converted ['ENSMUSG00000000001.5']


2026-11-24 11:54:20 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


sus_scrofa           configuration: local repository primary assembly 111  releases 56-115 | ENSSSCG00000000002 -> converted ['ENSSSCG00000000002.4']


2026-11-24 11:54:21 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


macaca_mulatta       configuration: local repository primary assembly 10   releases 48-115 | ENSMMUG00000000001 -> converted ['ENSMMUG00000000001.4']


2026-11-24 11:54:21 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


macaca_fascicularis  configuration: local repository primary assembly 6    releases 91-115 | ENSMFAG00000000004 -> converted ['ENSMFAG00000000004.2']


How to read the smoke test:
- `converted` with the id's own current version is the expected answer.
- `not_found` means the string did not match any identifier of the graph; `no_conversion` that it matched, but leads to no gene at the target release (`result.reason` names the cause, `API.reason_text(result)` says it in a sentence).
- `result.targets()` is a tuple because an answer can hold several targets (1→n); `result.candidates` holds every candidate, ranked.

## A.4 — Optional: deeper self-tests

`idtrack.checks.GraphChecks(graph, manager)` holds the self-tests of a graph: invariants of the graph alone, and comparisons of the graph with Ensembl's tables, which it reads through the database manager of the graph's tables (a check that reads a table downloads it on first use). Some of them are slow on large graphs. The command line runs them too: `idtrack verify <graph file> --repository <local repository>` ([Command line](../cli.rst)).

In [4]:
from idtrack.checks import GraphChecks

api = API.build('homo_sapiens', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
manager = API.database_manager('homo_sapiens', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
checks = GraphChecks(api.graph, manager)
# A small check, of the graph alone. Every check returns a result: `holds` says whether the invariant holds (it is also
# the truth value of the result), and `details` what was compared and every violation found.
result = checks.is_base_is_range_correct(progress=False)
print(result.check, '-> holds:', result.holds)
print('gene ids compared:', result.details['compared'], '| differing:', result.details['n_differing'])

2026-11-24 11:54:27 INFO:idtrack.build.database_manager: Ensembl releases of homo_sapiens assembly 38 read from 'idtrack_cache/ensembl_https_index_homo_sapiens_assembly-38.json' (scanned up to release 116).


is_base_is_range_correct -> holds: True
gene ids compared: 127247 | differing: 0


## A.5 — If something fails

1. **No externals configuration**: every supported organism ships one; for another organism, make one (Part 6). If a build warns that releases have no entry in the configuration, extend it (Part 6, section 6.6).
2. **Permission errors**: choose a local repository you can write to.
3. **Network or MySQL errors**: Ensembl REST and HTTPS access are needed to build; MySQL is optional (idtrack reads the HTTPS dumps where the ports do not answer; `IDTRACK_ENSEMBL_ACCESS=https` never tries them).
4. **Disk full**: the tables of a graph take from 0.24 GB (cynomolgus) to at most 9.3 GB (human): move your local repository to a larger disk ([Time and memory](../performance.rst)).